   ... normal to ... Computer architecture is notoriously dense, with a lot of
   moving parts between the software we write and the physical silicon that 
   executes it.

   ... high-level refresher of the first two lectures to help you get your 
   bearings again.


---   
LECTURE 1: ARCHITECTURE AND INSTRUCTIONS
   This lecture sets the stage for the entire module. It answers the fundamental
   question of why we care about hardware when we usually just write code, and
   introduces the bridge between the two.

   - THE BIG PICTURE: The core concept here is the INSTRUCTION SET ARCHITECTURE
     (ISA). The ISA is the critical interface or "contract" between software
     and hardware. It defines the set of instructions the processor can
     understand. 
   - WHY ARCHITECTURE MATTERS: The lecture motivates this by looking at modern
     accelerators. Standard CPU's aren't always enough anymore. The timestamps
     mention TPUs (Tensor Processing Units), which are highly specialised chips
     designed explicitly to accelerate matrix multiplications for machine
     learning models (like the Transformers powering GPT). Cloud providers
     (Microsoft, Amazon) rely heavily on these custom architectures.
   - THE RISC-V ARCHITECTURE: This module focuses on RISC-V (Reduced Instruction
     Set Computer). It is an open-source ISA that prioritises simplicity and
     efficiency.
      - R-TYPE INSTRUCTIONS: These are "Register-type" instructions used for
        standard arithmetic and logical operations (like `add`, `sub`, `and`).
        They typically take two source registers, perform an operation, and
        store the result in a destination register. 
      - FROM CODE TO SILICON: You looked at how high-level control flow (like an
        `if`-statement in C or C++) gets compiled down itno a sequence of these 
        basic RISC-V instructions (using branches and jumps).
      - THE DATAPATH: This is the physical wiring, ALUs, and memory components
        that actually execute the instructions. 





---
ISA: Instruction Set Architecture
   The ISA is the INTERFACE between software and hardware -- like an API for the
   processor. It defines:

   * What INSTRUCTIONS the processor understands (add, subtract, load, store,
     jump...)
   * What REGISTERS are available (small, fast storage inside the processor)
   * How MEMORY is addressed
   * The ENCODING FORMAT of instructions (how they're represented in binary)

   The key insight: DIFFERENT HARDWARE CAN IMPLEMENT THE SAME ISA. Your laptop
   and a server can both run the same RISC-V code, even if their internal
   circuits are completely different. This is what makes software portable.


DESIGN APPROACHES: RISC vs. CISC
   There are two major philosophies:


CISC (Complex Instruction Set Computer) -- e.g., x86 (Intel/AMD). Instructions
   can be very complex ("Multiply these two memory values and store the result
   back to memory in one instruction"). Instructions vary in length and can take
   many cycles.


RISC (Reduces Instruction Set Computer) -- e.g., RISC-V, ARM. Instructions are
   simple, fixed-length, and typically execute in one cycle. If you need 
   something complex, you combine multiple simple instructions. This makes the
   hardware much simpler and easier to pipeline (which you'll cover later).


   Your module uses RISC-V as the teaching architecture because it's clean, 
   open-source, and designed specifically to be easy to understand. 



---
RISC-V Architecture Basics
   RISC-V has 32 REGISTERS (x0 through x31), each 32 or 64 bits wide. Register
   x0 is hardwired to ALWAYS BE ZERO -- this is surprisingly useful (you'll
   see why). Instructions operate on registers, not directly on memory, 
   following the LOAD-STORE principle: you load data from memory into registers,
   do your computation, then store results back.


RISC-V R-type Instructions
   R-type instructions operate on THREE REGISTERS: two sources and one 
   destination. The format is:
```
add x5, x6, x7          # x5 = x6 + x7
sub x5, x6, x7          # x5 = x6 - x7
```

   The binary encoding has fixed fields:
```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```
   Every field has a fixed position -- this is what makes RISC decoding simple.
   The hardware doesn't have to "figure out" how long the instruction is or
   where the fields are. 

---

   ... they just look like alphabet soup until you break down what the 
   processor is actually doing with them. 

   ...

   breakdown of the R-Type instruction fields, reading from right to left (which
   is how the processor often starts looking at it):



---
- `opcode` (7 bits): This is the "Operation Code". It is the main category label
  for the instruction. When the processor reads these 7 bits, it says, "Ah, this
  is a standard Register-toRegister arithmetic operation" or "This is a memory
  load." For all R-Type arithmetic instructions, the opcode is exactly the same.

- `rd` (5 bits): This stands for REGISTER DESTINATION. This tells the processor
  where to save the final result of the calculation. It is 32 bits long because
  RISC-V has 32-general purpose registers, and it takes exactly 5 binary bits to
  count from 0 to 31.

- `funct3` (3 bits): This is a 3-bit "Function" code. Because the `opcode` only
  tells the processor the general category (e.g., "Arithmetic"), the `funct3`
  narrows it down further. For example, it might specify "This is an 
  addition/subtraction type of arithmetic" versus "This is a bitwise shift".

- `rs1` (5 bits): This stands for REGISTER SOURCE 1. It tells the processor
  which register holds the first input value for the math operation.

- `rs2` (5 bits): This stands for REGISTER SOURCE 2. It tells the processor 
  which register holds the second input value.

- `funct7` (7 bits): This is a 7-bit "Function" code, and it acts as the final
  tie-breaker. For example, the `add` and `sub` (subtract) instructions share
  the exact same `opcode` and the exact same `funct3`. The only way the
  processor knows whether to add or subtract is by looking at `funct7`.


WHY SPLIT IT UP LIKE THIS?
   In complex architectures like x86 (Intel/AMD), an instruction can be anywhere
   from 1 to 15 bytes long, and the fields move around. The hardware ahs to do a
   lot of complicated, slow work just to figure out what the instructions say.

   With RISC-V, the fields are in FIXED POSITIONS. When the 32-bit instructions
   hit the processor, the hardware wires can route bits 0-6 directly to the main
   control unit, and bits 7-11 directly to the register file's writee address, 
   all at the exact same time. It makes the physical silicon incredibly fast
   and simple to build. 


```
|  funct7 (7)  |  rs2 (5)  |  rs1  (5)  |  funct3 (3)  |  rd (5)  | opcode (7)  |
```

EXAMPLE: COMPILING IF-STATEMENTS
   This shows how high-level code maps to assembly. For example:

```
if (a == b) goto L1;
```

becomes:
```
beq x5, x6, L1          # branch to L1 if x5 == x6
```

   The key idea: EVERY high-level construct (if, while, for, function calls)
   eventually becomes a sequence of these simple instructions. The compiler's
   job is to do this translation efficiently.



---
LECTURE 2: PERFORMANCE    


...

#### 1. R-TYPE (Register)

WHAT IT IS
   R-type instruction takes TWO SOURCE REGISTERS, perform an operation, and 
   write the result to a DESTNATION REGISTER. No constants, no memory -- purely
   register-to-register.

WHEN TO USE R-TYPE
   When you're doing COMPUTATION BETWEEN TWO REGISTERS: `add`, `sub`, `and`, 
   `or`, `xor`, `sll` (shift left), `srl` (shift right), `slt` (set less than).
   These are the workhorses of any program.

WHEN NOT R-TYPE
   If one of your operand is a CONSTANT (like adding 5 to a register), you can't
   use R-type -- you need I-type. If you're accessing MEMORY, you need I-type
   (load) or S-type (store).



---
#### 2. I-TYPE (Immediate)

WHAT IT IS
   I-type insturctions use ONE SOURCE REGISTER and ONE CONSTANT VALUE EMBEDDED
   IN THE INSTRUCTION ITSELF (called an "immediate"). The result goes to a 
   destination register. 

   The encoding:
```
|  immediate (12)  |  rs1 (5)  |  funct3 (3)  |  rd (5)  |  opcode (7)  |
```
   Notice compared to R-type, the funct7 and rs2 fields are REPLACED by a single
   12-bit immediate. But rs1, funct3, and rd are in the EXACT SAME BIT POSITIONS
   as R-type -- this is deliberate, because it means the hardware that reads
   register numbers can be shared between instruction types.

   
TWO USE CASES SHOWN IN YOUR SLIDES
   USE CASE 1: Load from memory

```
ld x14, 8(x2)       # x14 = Memory[x2 + 8]
```
   
   Encoding:
```
| 000000001000 | 00010 | 010 | 01110 | 0000011 |
    imm = 8       x2     LW     x14    LOAD opcode
```

   How this works: the processor computes the EFFECTIVE ADDRESS by adding the
   immediate (8) to the value in rs1 (x2). Then it reads from that memory 
   address and stores the result in rd (x14).

   WHY THE OFFSET? In practice, x2 often points to the start of a data structure
   (like an array or a stack frame), and the immediate is the offset within that
   structure. For example, if x2 points to the base of a struct, `ld x14, 8(x2)`
   loads the field at byte offset 8.


USE CASE: Arithmetic with a constant
```
addi x15, x1, -50           # x15 = x1 - 50
```
         

Encoding:
```
 | 111111001110 | 00001 | 000 | 01111 | 0010011 |
    immm = -50     x1     ADD    x15    I-type opcode
```
   Notice the immediate is `111111001110` -- that's -5 in 12-BIT TWO'S 
   COMPLEMENT. The 12-bit value gets SIGN-EXTENDED to 32 bits before the
   addition, so negative numbers work correctly.

THE 12-BIT IMMEDIATE RANGE 
   With 12 bits (signed), you can encode constants from -2048 to +2047. This is
   enough for most offsets and small constants, but for larger values you'll 
   need U-tpe (see below).

WHEN TO USE I-TYPE
   Three situations: 
      * loading from memory (`ld`, `lw`, `lb`)
      * arithmetic with a small constant (`addi`, `andi`, `ori`, `slti`)
      * `jalr` -- jump to an address in a register plus offset

WHEN NOT TO USE I-TYPE
   If both operands are registers -> use R-type. If you need to STORE to memory
   --> use S-type (because stores need two register fields: the data and the
   base address, leaving no room for `rd`).

A SUBTLE BUT IMPORTANT POINT
   There's no `subi` instruction in RISC-V! If you want to subtract a constant,
   you just use `addi` with a NEGATIVE immediate: `addi x5, x6, -10` computes
   `x5 = x6 - 10`. This keeps the hardware simpler -- one fewer instruction to
   decode.



---
3. S-TYPE (Store)

WHAT IT IS
   S-type stores a register's value into MEMORY. It needs two register fields
   (the data to store AND the base address) plus an offset, but it DOESN'T
   WRITE TO ANY REGISTER -- so there's no `rd` field. 

THE ENCODING:
```
| imm[11:5] (7) | rs2 (5) | rs1 (5) | funct3 (3) | imm[4:0] (5) | opcode (7) |
```
   This is where it gets clever. The immediate is SPLIT INTO TWO PIECES: the 
   upper 7 bits go where `funct7` normally lives, and the lower 5 bits go where
   `rd` normally lives. Why? So that `rs1` and `rs2` remain in the EXACT SAME BIT
   POSITIONS as R-type and I-type. The hardware only needs one circuit to 
   extract register numbers, regardless of instruction type.

EXAMPLE FROM YOUR SLIDE:
```
sd x14, 8(x2)       # Memory[x2 + 8] = x14
```

Encoding:
```
| 0000000 | 01110 | 00010 | 010 | 01000 | 0100011 |
  imm[11:5]  x14     x2    SW   imm[4:0] STORE opcode
```
   The immediate 8 = `00000 01000`, split as: upper 7 bits = `0000000`, lower
   5 bits = `01000`. The processor reassembles these to get the offset 8, adds
   it to x2, and writes x14's value to that address.

WHEN TO USE S-TYPE
   Whenever you need to WRITE DATA TO MEMORY: `sd` (store doubleword/64-bit), 
   `sw` (store word/32-bit), `sh` (store halfword/16-bit), `sb` 
   (store byte/8-bit)   

```
ld  x14, 8(x2)   # I-type: Memory → Register  (x14 is the destination, rd)
sd  x14, 8(x2)   # S-type: Register → Memory  (x14 is the source, rs2)
```
   They look similar in assembly but COMPLETELY DIFFERENT FORMATS. Load has a 
   destination register (`rd`), so it fits I-type. Store has no destination
   register (it writes to memory, not a register), so it uses S-type with the
   split immediate. 
   


---
4. SB-TYPE / B-TYPE (Branch)
   
WHAT IT IS
   A variant of S-type used for CONDITIONAL BRANCHES (if-then jumps). It 
   compares two registers and jumps to a target address if the condition is
   met. 

EXAMPLE FROM YOUR SLIDE:
```
beq x19, x10, Label         # if x10 == x19, jump to Label
```
   The target address is encoded as an offset: PC + IMMEDIATE. In the slide, 
   Label is 16 bytes away, so the immediate encodes 16. 

THE ENCODING:
```
| imm[12] | imm[10:5] | rs2 | rs1 | funct3 | imm[4:1] | imm[11] | opcode |
```
   The immediate bits are SCRAMBLED across the instructions. This looks insa˜e,
   but there's a hardware reason: it keeps the most commonly needed bits in 
   positions that align with other instruction types, simplifying the circuit 
   that extracts them. 

   CRITICAL DETAIL: the immediate represents a BYTE OFFSET and BIT[0] is ALWAYS
   0 (because RISC-V instructions are always aligned to 2-byte boundaries). So
   the effective range is +- 4096 bytes from current PC -- roughly +- 1000 
   instructions.


AVAILABLE BRANCH CONDITIONS
   Using different funct3 values:

   * `beq` -- branch is equal
   * `bne` -- branch if not equal
   * `blt` -- branch if less than (signed)
   * `bge` -- branch if greater or equal
   * `bltu` -- branch if less than (unsigned)
   * `bgeu` -- branch if greater or equal (unsigned)


WHEN TO USE B-TYPE
   For implementing CONTROL FLOW, if-else, while loops, for loops. Every
   comparison-and-jump in your program becomes a B-type instruction. 



---
#### 5. U-TYPE (Upper Immediate)

THE PROBELM IT SOLVES
   I-type immediates are only 12 bits (-2048 to +2047). But what if you need to
   load a larger constant, like `0x87654000`? You can't fit that in 12 bits. 
   U-type solves this by loading a 20-bit value into the upper 20-bits of a 
   register, zeroing the lower 12 bits.

THE ENCODING:
```
| immediate [31:12] (20) | rd (5) | opcode (7) |
```
   This is the simplest format -- just a 20-bit immediate and a destination 
   register.


EXAMPLE FROM YOUR SLIDE
```
lui x10, 0x87654    # x10 = 0x87654000
```
   This loads `0x87654` into bits [31:12] of x10, setting bits [11:0] to zero.
   Result: `0x87654000`.


THE `lui` + `addi` PATTERN
   To load a full 32-bit constant like `0x87654321`:
```      
lui  x10, 0x87654       # x10 = 0x87654000  (upper 20 bits set)
addi x10, x10, 0x321    # x10 = 0x87654321  (lower 12 bits added)
```
   Step 1 sets the upper 20 bits. Step 2 fills in the lower 12 bits. Together,
   you can load any 32-bit value.

WHEN TO USE U-TYPE
   When you need LARGE CONSTANTS that don't fit in a 12-bit immediate. The
   compiler generates `lui` automatically whenever you use a large number.


---
6. UJ-Type / J-Type (Unconditional Jump)

WHAT IT IS
   Used for UNCONDITIONAL JUMPS -- "always go to this address, no questions
   asked." The main instruction is `jal` (Jump and Link).

Example from your slide:
```
jal x1, 32          # save PC+4 in x1, then jump to pC+32
```
   JUMP: The processor goes to PC + 32 (the target address).
   AND LINK: It also saves PC + 4 (the address of the next instruction) into
      x1. This is how FUNCTION CALLS work -- you jump to the function, and `x1`
      remembers where to come back to.

THE ENCODING
```
| imm[20] | imm[10:1] | imm[11] | imm[19:12] | rd (5) | opcode (7) |
```
   Like B-type, the immediate bits are scrambled for hardware reasons. The 
   20-bit immediate (with bit 0 always 0) gives a range of +-1 MB from the
   current PC.


WHEN TO USE J-TYPE
   For FUNCTION CALLS (`jal x1, function_label`) and UNCONDITIONAL JUMPS 
   (`jal x0, label`) -- saving the return address to x0 effectively discards it,
   making it a pure jump.


`jal` vs `jalr`
   `jal` (J-type) jumps to PC + offset -- the target is RELATIVE TO THE CURRENT
   INSTRUCTION. `jalr` (I-type) jumps to register + offset -- the target is 
   COMPUTED FROM A REGISTER VALUE. `jalr` is used for returning from functions:
   `jalr x0, 0(x1)` jumps to whatever address x1 holds.   

   In a branch instruction like `beq` (Branch if Equal), there actually is
   NO DESTINATION REGISTER. Unlike an `add` instruction that calculates a new 
   value and writes it to a register, a branch instruction is only interested in
   whether it should change the PROGRAM COUNTER (PC).

   In `beq x19, x10, Label`, the CPU compares the values in `x19` and `x10`. If
   they are equal, it adds the OFFSET (the `Label` value) to the CURRENT PROGRAM
   COUNTER (PC). So, the "destination" is the PC itself, not a general-purpose
   register like `x1`. This is why the instruction format replaces the `rd`
   (destination) field with pieces of the immediate values. 

   The immediate bits are "scrambled" across the instruction ... to keep the 
   `rs1` and `rs2` fields in the exact same position as they are in R-type
   instructions. This allows the hardware to start fetching the register values 
   before it even finishes decoding what kind of instruction it is!

---

7. COMPILING IF-STATEMENTS (Putting It All Together)
   The example from your slide:

```
if (i == j) f = g + h; else f = g - h;
```

   Register allocation: f->x19, g->x20, h->x21, i->x22, j->x23.


#### THE COMPILED CODE
```
        bne x22, x23, Else          # if i \neq j, skip to Else
        add x19, x20, x21           # f = g + h     (the "if" body)
        beq x0,  x0,  Exit          # unconditional jump to Exit
Else:   sub x19, x20, x21           # f = g - h     (the "else" body)
Exit:   ...
```


THE KEY TRICK: INVERT THE CONDITION
   The C code says `if (i == j)`. But the assembly uses `bne` (branch if NOT
   equal). Why?

   Think about how the processor executes: it goes LINE BY LINE from top to
   bottom. If the condition is true `(i == j)`, we want to FALL THROUGH to the
   next instruction (the add). If the condition is false `(i =/= j)`, we want
   to JUMP AWAY to the else body.

   So we branch on the OPPOSITE condition to skip over the if-body. This pattern
   is universal--you'll see it in every compiled if-statement.


THE UNCONDITIONAL JUMP
   `beq x0, x0, Exit` is a trick: since x0 always equals x0, this branch ALWAYS
   fires. It's effectively an unconditional jump. Without this, after executing
   the if-body (`add`), execution would FALL THROUGH into the else-body (`sub`)
   -- which would be wrong. The jump ensures only one branch executes.


---
QUICK REFERENCE SUMMARY

   Type // Purpose // Has `rd`? // Has immediate? // Example
   R // Register arithmetic // Yes // No // `add x5, x6, x7`
   I // Loads, arithmetic with constants // Yes // 12-bit // `addi x5, x6, x10`
   S // Store to memory // No // 12-bit (split) // `sd x5, 8(x6)`
   B // Conditional branch // No // 12-bit (split) // `beq x5, x6, Label`
   U // Load large upper constant // Yes // `lui x5, 0x12345`
   J // Unconditional jump // Yes // 20-bit // `jal x1, Label`

   The design philosophy is: keep REGISTER FIELDS IN THE SAME BIT POSITIONS
   across all formats, so the decoder hardware is simple. The immediate field
   reshapes itself to fit around those fixed register positions. 

---


   In the context of this module, is is exactly 32 BITS in each of the 32 bits
   registers. 

   This specific baseline architecture you are studying is called RV32I 
   (Risc-V 32-bit Integer).

   ... breakdown of how that looks hardware-wise:
      * THE NUMBER OF REGISTERS: You have exactly 32 general-purpose registers,
        named x0 through x31.
      * THE SIZE OF EACH REGISTER: Every single one of those registers holds
        exactly 32 BITS (which is 4 bytes) of data.

   In RV32,  a 32-bit chunk of data is officially called a "WORD". This is a 
   super important term to remember because it explains why everything else in
   the datapath lines up so perfectly. Your registers are 32 bits wide, your
   ALU processes 32 bits at a time, and your instructions are exactly 32 bits
   long. Everything is designed around that magic number to keep the hardware
   simple and fast!       



--- ---
   Registers. There are 32 registers. RISC-V names them x0 through x3. We're 
   using the 64-bit version of the RISC-V ISA, so each register holds a 64-bit
   value. 

---

## LECTURE 2: PERFORMANCE

### What is Performance?
Performance seems obvuious but is actually subtle. There are two main ways to
think about it:

   RESPONSE TIME (LATENCY): How long does one task take? ("My program finishes
   in 5 seconds")

   THROUGHPUT: How many tasks per unit time? ("The server handles 1000 requests
   per second")

   These can conflict -- a processor design that's great for throughput (like a
   GPU) might have terrible latency for a single task.


### The Performance Equation

This is the most IMPORTANT FORMULA in the module:
   `Execution Time = Instruction Count x CPI x Clock Period`

Or equivalently:
   `Execution Time = (Instruction Count x CPI) / Clock Rate`

Where:
   * INSTRUCTION COUNT = how many instructions the program executes (depends
     on the compiler and ISA)
   * CPI (Cycles per instruction) = Average number of clock cycles each 
     instruction takes (depends on the hardware design)
   * CLOCK PERIOD = duration of one cycle, or equivalently Clock Rate = cycles
     per second (depends on the hardware technology)

   This euqation is powerful because it separates the THREE INDEPENDENT FACTORS
   that determine speed. For example, CISC might have a lower instruction count
   (fewer, more powerful instructions) but higher CPI (each takes more cycles).
   RISC has more instructions but lower CPI.


EXAMPLE OF THE PERFORMANCE EQUATION
   If processor A runs at 2 GHz with CPI of 1.5, and processor B runs at 3 GHz
   with CPI of 2.0, which is faster for the same program?

      * A: time = IC x 1.5 / (2 x 10^9) = IC x 0.75 x 10
      * ...

   B is faster despite having worse CPI, because the clock rate more than
   compensates.
   


ASPECTS OF PROCESSOR PERFORMANCE
   You can improve performance by attacking any of the three factors:

   * REDUCE INSTRUCTION COUNT: better compiler, better ISA design
   * REDUCE CPI: pipelining, caches, branch prediction (covered in later 
     lectures)
   * INCREASE CLOCK RATE: better transistor technology, shorter critical paths  

   But there are TRADE-OFFS. Increasing clock rate often means more pipeline
   stages, which can increase CPI due to hazards. This is a recurring theme in
   architecture -- you're always balancing competing factors.



AMDAHL'S LAW
   A critical concept: if you speed up only PART of a program, the overall
   speedup is limited byt he part you DIDN'T speed up.

```
Overall Speedup = 1 / ((1 - f) + f/S)
```
   Where f = fraction of execution time affected, S = speedup of that
   fraction.

   Even with infinite speedup ($S \rightarrow \inf$), the best you can get is
   1/(1-f). If only 50% of execution benefits, maximum speedup is 2x, no matter
   what you do.



PERFORMANCE EVALUATION: SPEC Benchmarks
   How do you fairly compare processors? You an't just use clock speed (the
   "megahertz myth"). Instead, the industry uses standardised benchmark suites
   like SPEC (Standard Performance Evaluation Corporation). SPEC runs a set of
   real programs (compilers, simulations, games, etc.) and measures actual
   execution time, then computes a geometric mean ratio against a reference
   machine.

   The lecture shows SPEC CINT2006 results comparing an Opteron X4 to a Pentium
   4 -- despite the P4 having a higher clock rate, the Opteron wins on most
   benchmarks because of better CPI (better architecture, better caches, etc.)



TPU AND ACCELERATORS
   The lecture also touches on DOMAIN-SPECIFIC ACCELERATORS -- hardware designed
   for one specific task rather than general-purpose computing. Google's TPU
   (Tensor Processing Unit) is purpose-built for matrix multiplication used
   in neural networks. It sacrifices generality for massive performance gains
   on ML workloads. This connects to the broader theme: sometimes the best
   architecture depends entirely on what you're trying to compute.
   

### CPI: RISC vs. CISC

   ... Say you want to compute `A = Memory[100] + Memory[200]` and store the
   result back to `Memory[300]`.

   CISC APPROACH (x86-style): One complex instruction can do memory-to-memory
   operations directly.

```
ADD [300] [200] [100]      # 1 instruction, but takes ~5 cycles internally
```
   Instruction count = 1, CPI = 5, Total Cycles = 5.
   


---
RISC APPROACH (RISC-V)
```
lw x5, 100(x0)          # Load from memory         -> 1 cycle
lw x6, 200(x0)          # Load from memory         -> 1 cycle
add x7, x5, x6          # add registers            -> 1 cycle
sw x7, 300(x0)          # Store to memory          -> 1 cycle
```
   Instruction count = 4, CPI = 1. Total Cycles = 4.


---
   RISC is faster despite executing more instructions, because the low CPI more
   than compensates. In practice, RISC's simple instructions also enable higher
   clock rates (shorter $T$), widening the gap futher.

   The real insight: CPI and instruction counts are not independent -- choosing
   a simpler ISA (lower CPI) inevitably means more instructions, and vice versa.
   The performance equation helps you reason aabout which trade-off wins.



---
AMDAHL'S LAW: THE FULL INTUITION
   What $f$ and $S$ actually mean

   Imagine your PyTorch training loop takes 100 seconds per epoch. You profile 
   it and find:

   * 25 seconds = data loading, loss computation, gradient updates (runs on CPU,
     sequential).
   * 75 seconds = matrix multiplications in forward/backward pass (runs on GPU,
     PARALLELISABLE)

   So $f = 0.75$ (75% of the time is the part you can speed up) and 
   $(1 - f) = 0.25$ (25% is the sequential bottleneck you can't touch).

   Now you upgrade from a single GPU to setup that makes those matrix 
   multiplication $S = 10x$ faster (maybe you use tensor cores, better CUDA
   kernels, or multiple GPUs).

   The new time is:

```
New time = sequential part + (parallelizable part / S)
         = 25 + (75 / 10)
         = 25 + 7.5
         = 32.5 seconds
```
   Overall speed = 100 / 32.5 = 3.08x

   You made the GPU part 10x faster, but only got 3.08x overall. The 25 seconds
   of CPU work didn't budge. 


What happens as S -> $\inf$?
   ... the parallel part takes zero time. 

   ... maximum speedup = 100/25 = 4x

   That's the hard ceiling: $1 / (1 - f) = 1 / 0.25 = 4x$. No amount of GPU
   optimisation can beat this. The sequential 25% becomes 100% of your
   remaining runtime. 

   This is why NVIDIA and PyTorch invest heavily in things like `torch.compile`,
   CUDA graphs, and fused kernels for the data loading/gradient update parts too
   -- they're trying to increase $f$, not just $S$.


WHAT HAPPENS AS $f \rightarrow 0$?
   ... nearly all work is sequential.

   ...

   No speedup at all, regardless of $S$. Your fancy GPU sits idle.


---
WHAT HAPPENS AS $f \rightarrow 1$?
   If everything is parallelisable (f -> 1), the sequential bottleneck vanishes:

   ...

   You get the full speedup. If your CUDA kernel is 10x faster, the whole 
   program is 10x faster. This is the dream scenario -- pure GPU workloads like
   large batch inference come close to this. 



   ... ... 

---
THE PRACTICAL LESSON
   Amdahl's Law tells you WHERE TO FOCUS YOUR EFFORT. There's no point squeezing
   another $2 \times$ out of your CUDA kernel if the sequential bottleneck 
   dominates. Instead, you should attack the bottleneck -- use asynchronous data
   loading (`num_workers > 0`), overlap CPU and GPU work with `torch.cuda.Stream`
   , or move the optimiser step to the GPU with fused optimisers. These increase
   $f$, which raises the ciling for all future optimisations. 









SO WHAT IS AMDAHL'S LAW ACTUALLY SAYING?
   You're right that it's about diminishing returns, but the deeper point is
   about where your EFFORT HAS THE MOST LEVERAGE. There are three things you can
   do:

   * MAKE THE PARALLEL PART FASTER (increase $S$): Easy wins early, but 
     diminishing returns. Going from 1x to 10x is transformative. Going from
     10x to 100x barely matters. The curve flattens. 
   
   * MAKE THE SEQUENTIAL PART FASTER (shrink $1 - f$): Every improvement here
     raises the ceiling. This is where algorithmic breakthroughs like 
     AlphaEvolve matter enormously -- a 20% improvement to the bottleneck is 
     worth more than a 10x improvement to the already-fast part. 

   * CONVERT SEQUENTIAL WORK INTO PARALLEL WORK (increase $f$): This is the
     holy grail. If you can restructure your algorithm so that something that 
     was sequential becomes parallelisable, you're simultaneously shrinking the
     bottleneck and expanding the part that benefits from all your GPUs.

   * The third option is exatly what things like FlashAttention did. The 
     original attention mechanism had sequential memory access patterns that 
     bottlenecked on memory bandwidth (effectively sequential). FlashAttention
     restructured the algorithm to be more parallelism-friendly -- it didn't 
     just speed up $S$, it moved work from the $(1 - f)$ bucket into the $f$
     bucket. 


---
THE REAL-WORLD HIERARCHY OF IMPACT
   Think of it as a priority list:

   The most impactful thing is converting sequential --> parallel (changes $f$).
   This is what CUDA itself did for matrix operation -- they were once 
   CPU-sequential, and GPUs made them massively parallel.

   The second most impactful is algorithmic improvements to the sequential
   bottleneck (shrinks $1 - f$). This is your AlphaEvolve example -- finding a
   fundamentally better algorithm for the part that can't be parallelised.

   The least impactful (past a point) is throwing more hardware at the parallel
   portion (increases $S$). This is why buying a second RTX 5090 might give you
   far less than 2x improvement -- you're pushing against the ceiling. 

   Amdahl's Law isn't saying "parallelise or die." It's saying "KNOW WHERE YOUR
   CEILING IS, AND ATTACK THAT." Sometimes that means parallelising, sometimes
   it means a clever sequential algorithm, and often it means restructuring the
   problem entirely.



   Amdahl's Law, ... calculates the maximum theoretical speedup of a system when
   only a part of it is improved, highlighting that total performance gain is
   limited by the serial (non-parallelisable) portion. It proves that adding 
   more processors yields diminishing returns, as the bottleneck will always be
   non-parallelised code. 

KEY CONCEPTS AND FORMULAS

   * DEFINITION: The maximum improvement (speedup) expected when enhancing a
     portion of a system.
   * THE FORMULA: Speedup = $\frac{1}{(1 - P) + \frac{P}{S}}$     
      - $P$ = Parallelised fraction of the task (0 to 1).
      - $(1 - P)$ = Serial (non-parallelisable) fraction of the task. 
      - $S$ = Speedup of the parallelised portion.
   
   * KEY TAKEAWAY: ... 


KEY IMPLICATIONS
   * SERIAL BOTTLENECK: The non-parallel part of a program determines the upper
     bound of performance. 
   * DIMINISHING RETURNS: Adding vast numbers of processors ($N$) results in
     marginal improvements if the serial part is not minimised.
   * APPLICATION: Used in parallel computing, architecture design, and software
     optimiation to determine if enhancing a specific component is worth the
     effort.   


---

### LECTURE 3: MEMORY ADDRESSING AND ARCHITETURE EVOLUTION

1. LEVELS OF PARALLELISM
   The lecture opens by reinforcing a theme from Lecture 1: parallelism exists
   at EVERY LEVEL fo a modern computer, and the ISA is the bridge that lets 
   software exploit it.

   From top to bottom: your web browser fires off parallel REQUESTS (search, ads
   , video). The OS maps these to parallel THREADS, each assigned to a core. 
   Within each core, multiple INSTRUCTIONS execute simultaneously (pipelining --
   you'll cover this later). And at the lowest level, the ALU processes multiple
   DATA items in parallel (like adding 4 pairs of numbers at once - this is 
   SIMD).

   The key takeaway: hardware parallelism is useless unless software can express
   parallelism, and the ISA is what makes that possible. 


2. QUICK REVIEW OF INSTRUCTION TYPES
   This is revision from Lecture 1, but the lecture adds concrete binary 
   encodings. The widget above already covers this, and we went through all six
   types (R, I, S, B, U, J) in detail in ...

   One new detail from the transcript: WHY ARE LOAD (I-type) and STORE (S-type)
   DIFFERENT FORMATS? The RISC-V designers had a choice -- they could have made
   load and store the same format (like MIPS does). But they chose to keep
   register fields in consistent bit positions across all format. Since load
   needs a destination register (rd) and store needs a second source register
   (rs2), and these occupy different bit positions, they had to be different 
   types. The trade-off: slightly more complex instruction decoding, but the
   hardware for extracting register numbers is simpler and shared.    


3. MEMORY ADDRESSING (the new core material)
   This is the meat of the lecture. Play with the "Memory layout" and "Little
   endian" tabs abvoe, but here's the full explanation.

   WORDS AND BYTES. A "word" in RISC-V is 32-bits (4 bytes). A "double word" is
   64 bits (8 bytes). RISC-V is a 64-bit processor, meaning its ALU and 
   registers are 64 bits, but the term "word" still refers to 32 bits by 
   convention.

   BYTE-ADDRESSABLE MEMORY. Unlike some older architectures that give each word
   an address, RISC-V gives each byte a unique address. This means:

   * Word 0 occupies byte addresses 0, 1, 2, 3
   * Word 1 occupies byte addresses 4, 5, 6, 7
   * Word 2 occupies byte addresses 8, 9, 10, 11

   Since each word spans 4 bytes, WORD ADDRESSES INCREMENT BY 4, not by 1. This 
   is why you see `lw x3, 0(x2)` then `lw x4, 4(x2)` to load consecutive words
   -- the offset jumps by 4. 

   This also applies to instructions: every RISC-V instruction is 32 bits (4
   bytes), so the PC increments by 4 after each instruction. When a branch says
   "jump by 16 bytes", that means jump over 4 instructions.

   LITTLE ENDIAN. When a 4-byte value like `0x01EE2842` is stored in memory
   starting at address 8, which byte goes where? In little endian (which RISC-V
   uses), the LEAST SIGNIFICANT BYTE goes at the LOWEST ADDRESS:

      * Address 8: `0x42` (least significant)
      * Address 9: `0x28`
      * Address 10: `0xEE`
      * Address 11: `0x01` (most significant)

   This might seem backwards, but it has practical advantages: if you read just
   one byte from address 8 (`lb` instruction), you get least significant byte --
   which is the value modulo 256. This is useful in many algorithms. 

   LOAD INSTRUCTIONS:
   * `lw` -- load word (32 bits)
   * `ld` -- load double word (64 bits)
   * `lb` -- load byte (8 bits)



4. ADDRESSING MODES
   These are the four ways RISC-V accesses data (see the "Addressing modes" tab
   above):

   REGISTER ADDRESSING -- operands are in registers. This is the fastest because 
   registers are physically inside the processor. Used by R-type instructions
   like `add x5, x6, x7`.

   IMMEDIATE ADDRESSING -- one operand is a constant in the instruction itself. 
      No memory access needed. Used by I-type instructions like 
      `addi x5, x6, 42`.

   BASE ADDRESSING -- the memory address is computed as register + offset. This 
      is how loads and stores work: `lw x5 8(x2)` means "go to address x2+8 in 
      memory." The register typically holds a base pointer (start of an array,
      struct, or stack frame), and the immediate is the offset within it. 

   PC-RELATIVE ADDRESSING -- the address is computed as PC + offset. Used by
      branches and jumps. This is clever because it means branch targets are
      relative to the current instruction, so the code is POSITION-INDEPENDENT
      -- it works regardless of where in memory it's loaded.   



---
5. RISC vs. CISC: The Concrete Comparison
   The lecture compares MIPS (RISC) with the Motorola 68000 (CISC) using real
   numbers.

   The 68000 has 8 data registers and 8 address registers (you must keep track
   of which is which), 12 addressing modes, and complex instructions. MIPS has 
   32-general-purpose registers and simple, uniform instructions. 


---
THE BENCHMARK RESULTS:
   * MIPS (R4400) is 4.2x faster for integer and 6.5x faster for floating point
   * But MIPS costs 4.7x more

   So MIPS is LESS COST-EFFECTIVE for integers (4.2x performance for 4.7x cost)
   but more COST-EFFECTIVE for floats (6.5x for 4.7x cost). The lesson: raw
   speed isn't everything -- cost per unit of performance matters.

   THE CODE DENSITY PROBLEM: The lecture shows a concrete example where the 
   68000 does an array accumulation in a single 16-bit instruction: 
   `add.l d0, (a0)+` -- this adds the value at address a0 to d0 AND
   auto-increments a0, all in one instruction.

   MIPS needs three 32-bit instructions to do the same thing (load, add pointer,
   add value) -- that's 96 bits vs 16 bits. RISC code is 6x larger in this case.
   This matters for embedded systems with limited memory (like your phone), and
   it's one reason ARM (a RISC architecture) added a "Thumb" mode with shorter
   16-bit instructions. 


---
6. ARCHITECTURE CLASSIFICATION
   See the "Architcture types" tab above. the    




ARCHITECTURE CLASSIFICATION BY MEMORY ACCESS
   Three ways to organise where temporary values live during computation.


TASK: compute C = A + B

   STACK MACHINE
   push A
   push B
   add
   pop C

   - Operands are implicit (top of stack). Very compact instructions. But
     inflexible -- no random access to deeper stack elements. 


   ACCUMULATOR MACHINE
   load A
   add B
   store C

   - One operand is always the single accumulator register. Short instructions.
     But frequent memory access since there's only one register. 

   REGISTER MACHINE (RISC-V)
   load R1, A
   add R2, R1, B
   store C, R2

   - All operands are explicit registers. Fast (registers are faster than
     memory). Longer instructions since you must name every operand. This is 
     what RISC-V uses.
       


6. ARCHITECTURE CLASSIFICATION
   ... The three types represent an evolution:

   STACK MACHINES (like the early B5500, or the Java Virtual Machine) -- 
   operands are implicitly on top of the stack. Super compact instructions. But
   you can't randomly access data deep in the stack, and if the stack lives in
   memory, it's slow.

   ACCUMULATOR MACHINES (like early Intel 8080, MOS 6502) -- one register, 
   called the accumulator. All arithmetic goes through it. Short instructions
   (only need to specify one memory address). But you're constantly shuffling
   data between the accumulator and memory. 

   REGISTER MACHINES (like RISC-V, ARM, x86) -- multiple general-purpose 
   registers. Fast because register access is much faster than memory. But
   instructions are longer since you must name all operands explicitly. 

   The historical progression went: accumulator (cheap, 1950s) -> stack (1960s,
   compact code) -> general purpose registers (1970s+, fast). Then within
   register machines: CISC (complex instructions, 1970s-80s) -> RISC (simple
   instructions, 1980s+) -> multi-core (2005+, because clock speeds hit the
   power wall).



7. THE END OF DENNARD SCALING
   This is the "big picture" sldie. Until ~2005, you could make processors
   faster by simply shrinking transistors and increasing clock speed. This 
   worked because of DENNARD SCALING -- as transistors get smaller, their
   power consumption drops proportionally, so you can run them faster without 
   overheating.

   Around 2005, Dennard scaling broke down. Transistors got so small that
   LEAKAGE CURRENT (power wasted even when transistors are "off") become
   dominant. The result: you couldn't increase clock speed anymore without the
   chip melting.

   The industry's response: STOP MAKING SINGLE CORES FASTER, START ADDING MORE
   CORES. This is why we went from single-core processors to 4-core, 8-core, and
   eventually the thousands-of-cores GPUs you use for ML. It's also why Amdahl's
   Law became so criticaly important -- if you code can't use multiple cores,
   you're stuck.



8. CUSTOM INSTRUCTIONS AND AMDAHL'S LAW
   The lecture ends by connecting Amdahl's Law to custom instructions. If you
   frequently compute $x^2 + y^2$, you could add a dedicated `sumsq` instruction
   . But Amdahl's Law warns you: even if this instruction makes 90% of your 
   runtime 100x faster, the overall speedup is only 9.17x -- because the remaining
   10% becomes the bottleneck.

   This is the same principle behind domain-specific architectures like Google's 
   TPU or NVIDIA'S tensor cores. They're spectacularly fast at matrix 
   multiplication, but only useful if your workload is dominated by matrix
   multiplication. For general-purpose computing, the silicon spend on 
   specialised hardware is wasted.

---

Overall Speedup = 1 / ((1 - f) + f/S)

---

   ... (1) trace through a 4-bit addition on the ripple carry adder to see how 
   carry propagates, and (2) trace the ALU control signals for each operation to
   verify the truth table. I'll build you an interactive ALU you can play with,
   then explain everything.


   SLT (s=0, d_1=1, d_0=1): First, it computes A - B using subtraction (invert B
   , carry-in=1). Then it checks the MSB of the result: if MSB=1, the result
   is negative, meaning A < B. 


---   
### LECTURE 4: ALU (Arithmetic Logic Unit)

1. Number Representation Review
   Before building the ALU, the lecture reminds you of the foundation it 
   operates on.

   TWO'S COMPLEMENT is how signed integers are represented. The MSB (most
   significant bit) has a negative weight. ...

   ...

   For $n$ bits, the range is -2^(n-1) to 2^(n-1)-1. So 4 bits gives -8 to +7,
   and 32 bits give roughly -2.1 billion to +2.1 billion.

   SIGN EXTENSION means copying the MSB when you widen a number, `1011` (-5 in
   4-bit) becomes `1111011` (still -5 in 7-bit). The new bits are all copies of
   the sign bit. This is why the I-type immediate in RISC-V gets "sign-extended"
   from 12 bits to 32/64-bits -- it preserves the value whether positve or
   negative.

   OVERFLOW happens when the result doesn't fit. The rules are simple: if you
   add two positives and get a negative, or add two negatives and get a 
   positive, something went wrong. The ALU needs to detect this.


2. LOGICAL OPERATIONS
   Before building the ALU, the lecture catalogs what it needs to support.

   SHIFTS move all bits left or tight by some amount. `slli` (shift left logical
   immediate) `slli x10, x16, 8` shifts x16 left by 8 bits, filling the empty
   positions with zeros. Left shift by n is equivalent to multiplying by 2^n
   (think about it: shifting `0011` (3) left by 1 gives `0110` (6) -- that's
   x2). Right shift by n divides by 2^n.

   There's a subtlety with right shifts: `srl` (shift right logical) fills the
   top with zeros, while `sra` (shift right arithmetic) copies the sign bit. 
   This matters for negative numbers -- arithmetic right shift preserves the 
   sign, so -8 >> 1 = -4 (correct division by 2), while logical right shift 
   would give a large positive number.

   BITWISE OPERATIONS (`and`, `or`) operate on each pair of corresponding
   bits independently. These are important for masking (isolating specific bits)
   , setting flags, and clearing bits -- you used `band` etensively in your
   FunnyCore division code.


3. BUILDING BLOCKS: The Ground Floor
   The ALU needs to compute AND, OR, ADD and SUB. The lecture builds it from the
   smallest pieces upward.

   INVERTER:    








---

   The `slt` (Set Less Than) command in MIPS assembly is a comparison 
   instruction that sets a destination register to 1 if the first operand is 
   less than the second, and 0 otherwise. It is used to translate high-level
   inequalities (`<`, `>`) into assembly, usually followed by `beq`, `bne` to 
   branch based on the result.


SYNTAX AND OPERATION
   * SYNTAX: `slt $rd, $rs, $rt` (Set rd to 1 if rs < rt, else 0).
   * MECHANISM: It compares the signed values of two registers (`$rs` and `$rt`)
     and stores the result in a destination register (`$rd`).
   * EXAMPLE: `slt $t0, $s0, $s1` sets `$t0` to 1 if register `$s0` is less than
     `$s1`, or 0 otherwise.

KEY DETAILS
   * VARIANTS: `slti` (Set Less Than Immediate) compares a register with a 
     16-bit constant. `sltu` performs the same operation for unsigned numbers.
   * USE IN CONTROL FLOW: Because `slt` only sets a register, it is commonly
     paired with branch instructions.
      * Example: To implement `if ($s0 < $s1) { ... }`, the assembly would be:

```assembly
slt $t0, $rs0, $rs1             # $t0 = 1 if $rs0 < $rs1
bne $t0, $zero, label           # Branch if $t0 is not 0
```
